# Restrições de string

Assim como as restrições numéricas do `Field`, há restrições aplicáveis a strings — e, na verdade, a tipos de sequência em geral, não só
strings, embora a documentação não pareça mencionar isso.

São elas:
- `min_length` -> tamanho mínimo da string (ou sequência)
- `max_length` -> tamanho máximo da string (ou sequência)

(Documentação [aqui](https://docs.pydantic.dev/latest/api/fields/).)

In [1]:
from pydantic import BaseModel, Field, ValidationError

In [2]:
class Model(BaseModel):
    name: str = Field(min_length=1, max_length=20)

In [3]:
print(f'Model(name="Richard"): {Model(name="Richard")!r}')

Model(name="Richard"): Model(name='Richard')


In [4]:
try:
    Model(name="")
except ValidationError as ex:
    print(f"{type(ex).__name__}: {ex}")

ValidationError: 1 validation error for Model
name
  String should have at least 1 character [type=string_too_short, input_value='', input_type=str]
    For further information visit https://errors.pydantic.dev/2.13/v/string_too_short


In [5]:
try:
    Model(name="*" * 21)
except ValidationError as ex:
    print(f"{type(ex).__name__}: {ex}")

ValidationError: 1 validation error for Model
name
  String should have at most 20 characters [type=string_too_long, input_value='*********************', input_type=str]
    For further information visit https://errors.pydantic.dev/2.13/v/string_too_long


Essas restrições também valem para tipos de sequência, como listas e tuplas:

In [6]:
class Model(BaseModel):
    items: list[float] = Field(min_length=3, max_length=5, default=[1.0, 2.0, 3.0])

In [7]:
print(f"Model(): {Model()!r}")

Model(): Model(items=[1.0, 2.0, 3.0])


In [8]:
print(f"Model(items=[2, 3.5, 4]): {Model(items=[2, 3.5, 4])!r}")

Model(items=[2, 3.5, 4]): Model(items=[2.0, 3.5, 4.0])


In [9]:
try:
    Model(items=[1, 2])
except ValidationError as ex:
    print(f"{type(ex).__name__}: {ex}")

ValidationError: 1 validation error for Model
items
  List should have at least 3 items after validation, not 2 [type=too_short, input_value=[1, 2], input_type=list]
    For further information visit https://errors.pydantic.dev/2.13/v/too_short


Podemos até usar essa abordagem (veremos em breve uma forma um pouco mais simples) para voltar a um exemplo anterior: a classe `Circle` em
que o `center` precisava ser uma tupla de 2 ou 3 elementos.

Acabamos usando `conlist`, o que nos obrigou a aceitar uma lista em vez de uma tupla.

Podemos usar a abordagem acima, desde que consigamos declarar uma tupla de tamanho variável — ou seja, indicar o tipo dos elementos e que
pode haver quantos quisermos (e depois restringimos com `Field`).

Isto **não** funciona, porque o *type hint* indica que a tupla só pode ter **um** elemento inteiro:

In [10]:
class Circle(BaseModel):
    center: tuple[int]
    radius: int = Field(gt=0, default=1)

In [11]:
print(f"Circle(center=(1,), radius=1): {Circle(center=(1,), radius=1)!r}")

Circle(center=(1,), radius=1): Circle(center=(1,), radius=1)


In [12]:
try:
    Circle(center=(1, 1), radius=1)
except ValidationError as ex:
    print(f"{type(ex).__name__}: {ex}")

ValidationError: 1 validation error for Circle
center
  Tuple should have at most 1 item after validation, not 2 [type=too_long, input_value=(1, 1), input_type=tuple]
    For further information visit https://errors.pydantic.dev/2.13/v/too_long


O *type hint* de tuplas não funciona como o de listas — basta indicar que a tupla tem tamanho variável:

In [13]:
class Circle(BaseModel):
    center: tuple[int, ...]
    radius: int = Field(gt=0, default=1)

In [14]:
print(f"Circle(center=(1, 1)): {Circle(center=(1, 1))!r}")

Circle(center=(1, 1)): Circle(center=(1, 1), radius=1)


In [15]:
print(f"Circle(center=(1, 1, 1)): {Circle(center=(1, 1, 1))!r}")

Circle(center=(1, 1, 1)): Circle(center=(1, 1, 1), radius=1)


In [16]:
print(f"Circle(center=(1, 2, 3, 4, 5)): {Circle(center=(1, 2, 3, 4, 5))!r}")

Circle(center=(1, 2, 3, 4, 5)): Circle(center=(1, 2, 3, 4, 5), radius=1)


Parece funcionar; agora só falta restringir o tamanho da tupla:

In [17]:
class Circle(BaseModel):
    center: tuple[int, ...] = Field(min_length=2, max_length=3, default=(0, 0))
    radius: int = Field(gt=0, default=1)

In [18]:
print(f"Circle(): {Circle()!r}")

Circle(): Circle(center=(0, 0), radius=1)


In [19]:
print(f"Circle(center=(1, 2, 3)): {Circle(center=(1, 2, 3))!r}")

Circle(center=(1, 2, 3)): Circle(center=(1, 2, 3), radius=1)


In [20]:
try:
    Circle(center=(1, ))
except ValidationError as ex:
    print(f"{type(ex).__name__}: {ex}")

ValidationError: 1 validation error for Circle
center
  Tuple should have at least 2 items after validation, not 1 [type=too_short, input_value=(1,), input_type=tuple]
    For further information visit https://errors.pydantic.dev/2.13/v/too_short


In [21]:
try:
    Circle(center=(1, 2, 3, 4))
except ValidationError as ex:
    print(f"{type(ex).__name__}: {ex}")

ValidationError: 1 validation error for Circle
center
  Tuple should have at most 3 items after validation, not 4 [type=too_long, input_value=(1, 2, 3, 4), input_type=tuple]
    For further information visit https://errors.pydantic.dev/2.13/v/too_long


Outro recurso do `Field`, este só para strings (não sequências em geral), é a restrição por **expressão regular**.

O curso não entra em regex (o autor conhece o básico e acha o resto confuso).

Mas, se você conhece regex, veja como usá-la em restrições de string.

Especificamos uma regex com o argumento `pattern` do `Field`.

In [22]:
class Model(BaseModel):
    zip_code: str = Field(pattern=r"^[0-9]{5}(?:-[0-9]{4})?$")

In [23]:
print(f'Model(zip_code="12345"): {Model(zip_code="12345")!r}')

Model(zip_code="12345"): Model(zip_code='12345')


In [24]:
print(f'Model(zip_code="12345-1234"): {Model(zip_code="12345-1234")!r}')

Model(zip_code="12345-1234"): Model(zip_code='12345-1234')


In [25]:
try:
    Model(zip_code="12345-12345")
except ValidationError as ex:
    print(f"{type(ex).__name__}: {ex}")

ValidationError: 1 validation error for Model
zip_code
  String should match pattern '^[0-9]{5}(?:-[0-9]{4})?$' [type=string_pattern_mismatch, input_value='12345-12345', input_type=str]
    For further information visit https://errors.pydantic.dev/2.13/v/string_pattern_mismatch


> 💡 **Complemento (contexto brasileiro):** o mesmo para CEP, aceitando com ou sem hífen. Lembre que `pattern` valida, mas não normaliza —
> `"01310-100"` e `"01310100"` continuam diferentes; para normalizar, use um validador (seção 8).

In [26]:
class Endereco(BaseModel):
    cep: str = Field(pattern=r"^\d{5}-?\d{3}$")

print(f'[Endereco(cep=c).cep for c in ("01310-100", "01310100")]: {[Endereco(cep=c).cep for c in ("01310-100", "01310100")]!r}')

[Endereco(cep=c).cep for c in ("01310-100", "01310100")]: ['01310-100', '01310100']
